# VisionBrain End-to-End: da captura ao dashboard

Tutorial reproduzível para pessoas desenvolvedoras e pesquisadoras de visão computacional. Ele percorre aquisição sintética, qualidade de imagem, preprocessamento, tracking simulado, regras, eventos, evidências e leitura pelo dashboard — sem webcam, download de pesos ou serviço externo.

**Pré-requisitos:** Python 3.11+, dependências base do projeto e o repositório aberto na raiz.

**Ao final você saberá:** validar o caminho operacional completo e localizar os artefatos consumidos pelo dashboard.

## Roteiro

1. Preparar imports e diretórios.
2. Capturar um frame determinístico.
3. Medir qualidade e aplicar preprocessamento.
4. Executar o fluxo E2E sintético.
5. Inspecionar eventos e evidências.
6. Explorar um exercício de configuração.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook_e2e'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Projeto: {PROJECT_ROOT}')
print(f'Artefatos: {OUTPUT_DIR}')

## 1. Aquisição sem hardware

A fonte sintética implementa o mesmo contrato usado pelo runtime. Isso torna testes, CI e demonstrações independentes de driver, câmera e rede.

In [ ]:
from IPython.display import display
from PIL import Image
import cv2

from visionbrain.camera.synthetic_source import SyntheticCamera
from visionbrain.config import CameraConfig

camera_config = CameraConfig(source='synthetic', width=320, height=180, warmup_frames=0)
with SyntheticCamera(camera_config) as camera:
    captured = camera.read()
    probe = camera.probe()

frame = captured.frame
print(json.dumps(probe, indent=2))
display(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))

## 2. Quality gate e preprocessamento

As métricas individuais são sinais operacionais; o score agregado não deve ser tratado como medida universal. A receita real precisa de thresholds calibrados para a tarefa.

In [ ]:
from visionbrain.config import PreprocessConfig
from visionbrain.preprocess.pipeline import PreprocessPipeline
from visionbrain.quality.advisor import recommendations
from visionbrain.quality.metrics import analyze_frame_quality

quality = analyze_frame_quality(frame)
processed = PreprocessPipeline(PreprocessConfig(adaptive=True)).apply(frame, quality)
print(json.dumps(quality.to_dict(), indent=2))
print('Recomendações:')
for item in recommendations(quality):
    print('-', item)
display(Image.fromarray(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)))

## 3. Pipeline completo

O demo move uma peça com `track_id` estável por uma zona de inspeção. Ele produz `object_entered`, `zone_enter` e `zone_exit`, grava o audit trail em JSONL e salva snapshots anotados.

In [ ]:
from visionbrain.demo import run_synthetic_e2e

summary = run_synthetic_e2e(OUTPUT_DIR, frames=24)
print(json.dumps(summary, indent=2))
assert summary['frames_processed'] == 24
assert summary['event_types'].get('zone_enter', 0) >= 1
assert summary['event_types'].get('zone_exit', 0) >= 1

## 4. Contrato consumido pelo dashboard

O dashboard não controla a câmera: ele lê o audit trail persistido e resolve evidências locais. Essa separação mantém a UI fora do loop de inferência.

In [ ]:
from visionbrain.dashboard.data import flatten_event, load_events, summarize_events

events, malformed = load_events(summary['events_path'])
dashboard_summary = summarize_events(events)
rows = [flatten_event(event) for event in events]
print(json.dumps(dashboard_summary, indent=2))
print(f'Linhas inválidas: {malformed}')
rows[:3]

In [ ]:
snapshot_paths = [Path(row['snapshot']) for row in rows if row.get('snapshot')]
print(f'{len(snapshot_paths)} evidências encontradas')
if snapshot_paths:
    snapshot = snapshot_paths[0]
    if not snapshot.is_absolute():
        snapshot = PROJECT_ROOT / snapshot
    display(Image.open(snapshot))

## Exercício

Reduza a zona para o terço central da imagem e preveja como isso altera o momento dos eventos `zone_enter` e `zone_exit`. Depois execute a célula de apoio abaixo e use os pontos na sua própria `ZoneConfig`.

In [ ]:
# Apoio/resposta: zona vertical central mais estreita.
central_third = [(0.33, 0.15), (0.66, 0.15), (0.66, 0.85), (0.33, 0.85)]
print('Use estes pontos em ZoneConfig:', central_third)
print('Esperado: entrada mais tarde e saída mais cedo, pois a área útil ficou menor.')

## Próximos passos e armadilhas

- Abra o dashboard com `visionbrain dashboard` e selecione `outputs/notebook_e2e/events.jsonl`.
- Para webcam real, troque a fonte e valide propriedades retornadas pelo driver; `set()` não garante aplicação.
- Para YOLO, instale o extra `yolo`; o notebook evita download de pesos de propósito.
- Não use um VLM em cada frame. Enriqueça snapshots de eventos fora do loop crítico.
- GenICam, GPU, RTSP e benchmarks de hardware só ficam validados depois de testes no dispositivo final.